# 거래 가능 코인 일봉 (Binance 공개 API)
- 종목 목록: `tradable_assets.md`의 **코인 현물 (USDT)** 20개 + **무기한 선물** 6개
- 기간: 1996-01-01 요청 → 실제로는 각 심볼의 바이낸스 상장일부터 (API 키 불필요)
- 일봉 기준: **UTC 00:00 ~ 23:59** (날짜 = 캔들 시작일, 한국시간 09:00 시작). 아직 마감 안 된 오늘 캔들은 제외
- 코인은 액면분할·배당이 없어 원주가 = 수정주가 (조정 불필요)

| 구분 | 엔드포인트 | 1회 최대 |
|---|---|---|
| 현물 | `api.binance.com/api/v3/klines` | 1000개 |
| USD-M 선물 (`.P`) | `fapi.binance.com/fapi/v1/klines` | 1500개 |
| COIN-M 선물 (`_PERP`) | `dapi.binance.com/dapi/v1/klines` | 1500개, **기간 최대 200일** |

In [ ]:
import os
import re, time
from pathlib import Path
import pandas as pd
import requests

PROJECT_DIR = Path(os.environ.get("FP2_PROJECT_DIR", Path.cwd()))   # 데이터·결과 폴더 (환경변수 FP2_PROJECT_DIR로 지정, 기본=노트북 폴더)
MD_PATH = PROJECT_DIR / "tradable_assets.md"
START = pd.Timestamp("1996-01-01", tz="UTC")
NOW = pd.Timestamp.now(tz="UTC")
DAY_MS = 86_400_000

## 1. md 파일에서 심볼 추출

In [ ]:
text = MD_PATH.read_text(encoding="utf-8")
def section(title):
    return re.search(rf"^## {title}.*?$(.*?)(?=^## |\Z)", text, flags=re.M | re.S).group(1)

spot_sec = section("코인 현물")
spot = {m.group(1): m.group(2).strip() for m in re.finditer(r"\b([A-Z0-9]+USDT)\s+([^/\n]+)", spot_sec)}

fut_sec = section("코인 무기한 선물")
usdm = re.findall(r"\b([A-Z]+USDT)\.P\b", fut_sec)      # 'BTCUSDT.P' → API 심볼 'BTCUSDT'
coinm = re.findall(r"\b([A-Z]+USD_PERP)\b", fut_sec)

print(len(spot), "현물:", list(spot))
print("USD-M:", usdm, "| COIN-M:", coinm)

## 2. 일봉 다운로드 함수 (페이지네이션)

In [ ]:
MARKETS = {
    "spot":  dict(url="https://api.binance.com/api/v3/klines",  limit=1000, max_days=None),
    "usdm":  dict(url="https://fapi.binance.com/fapi/v1/klines", limit=1500, max_days=None),
    "coinm": dict(url="https://dapi.binance.com/dapi/v1/klines", limit=1500, max_days=200),
}
COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
        "quote_volume", "trades", "taker_buy_base", "taker_buy_quote", "ignore"]

def get(url, params, retries=5):
    for i in range(retries):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (418, 429):          # 레이트리밋 → 대기
            time.sleep(int(r.headers.get("Retry-After", 2 ** i)))
        else:
            r.raise_for_status()
    raise RuntimeError(f"failed: {url} {params}")

def fetch_klines(symbol, market):
    m = MARKETS[market]
    start, end_all = int(START.timestamp() * 1000), int(NOW.timestamp() * 1000)
    rows = []
    while start < end_all:
        params = dict(symbol=symbol, interval="1d", startTime=start, limit=m["limit"])
        if m["max_days"]:                           # COIN-M: 200일 창 단위로 이동
            params["endTime"] = min(start + m["max_days"] * DAY_MS - 1, end_all)
        data = get(m["url"], params)
        if data:
            rows += data
            start = data[-1][0] + DAY_MS
        elif m["max_days"]:
            start = params["endTime"] + 1           # 상장 전 빈 구간 → 다음 창
        else:
            break                                   # 현물/USD-M: 더 이상 데이터 없음
        time.sleep(0.1)

    df = pd.DataFrame(rows, columns=COLS).drop(columns="ignore")
    df = df[df["close_time"] < end_all]            # 마감 안 된 오늘 캔들 제외
    df["date"] = pd.to_datetime(df["open_time"], unit="ms").dt.normalize()
    num = ["open", "high", "low", "close", "volume", "quote_volume", "taker_buy_base", "taker_buy_quote"]
    df[num] = df[num].astype(float)
    return df.drop_duplicates("date").set_index("date")[num + ["trades"]]

## 3. 전체 다운로드

In [ ]:
jobs = [(s, "spot", s) for s in spot] + \
       [(s, "usdm", f"{s}.P") for s in usdm] + \
       [(s, "coinm", s) for s in coinm]

data, failed = {}, {}
for sym, market, label in jobs:
    try:
        data[label] = fetch_klines(sym, market)
        d = data[label]
        print(f"{label:14s} {market:5s} {d.index.min().date()} ~ {d.index.max().date()}  {len(d):5d}일")
    except Exception as e:
        failed[label] = repr(e)
print("실패:", failed)

## 4. 종가 wide 테이블 + 요약

In [ ]:
spot_close = pd.DataFrame({s: data[s]["close"] for s in spot if s in data}).sort_index()
perp_labels = [f"{s}.P" for s in usdm] + coinm
perp_close = pd.DataFrame({s: data[s]["close"] for s in perp_labels if s in data}).sort_index()
spot_close.index.name = perp_close.index.name = "date"

names = {**spot, **{f"{s}.P": f"USD-M 무기한 {s}" for s in usdm}, **{s: f"COIN-M 무기한 {s}" for s in coinm}}
summary = pd.DataFrame({
    "name": [names[k] for k in data],
    "market": [m for _, m, l in jobs if l in data],
    "first_date": [d.index.min() for d in data.values()],
    "last_date":  [d.index.max() for d in data.values()],
    "n_days": [len(d) for d in data.values()],
    "missing_days": [(d.index.max() - d.index.min()).days + 1 - len(d) for d in data.values()],
}, index=pd.Index(list(data), name="symbol"))
summary.sort_values("first_date")

## 5. 검증: 현물 vs 무기한 선물 괴리, 이상 수익률

In [ ]:
for s in usdm:
    gap = (perp_close[f"{s}.P"] / spot_close[s] - 1).dropna()
    print(f"{s}.P vs 현물: 평균 괴리 {gap.mean():+.3%}, 최대 |괴리| {gap.abs().max():.2%}")

r = pd.concat([spot_close, perp_close], axis=1).pct_change(fill_method=None).stack()
r[r.abs() > 0.5].sort_values()   # 일간 ±50% 초과

## 6. 저장

In [ ]:
spot_close.to_csv(PROJECT_DIR / "crypto_spot_close.csv")
perp_close.to_csv(PROJECT_DIR / "crypto_perp_close.csv")
ohlcv = pd.concat(data, names=["symbol"]).reset_index()      # 전체 OHLCV (long 형식)
ohlcv.to_csv(PROJECT_DIR / "crypto_ohlcv_1d.csv", index=False)
summary.to_csv(PROJECT_DIR / "crypto_summary.csv", encoding="utf-8-sig")
print("saved:", spot_close.shape, perp_close.shape, ohlcv.shape)